# EDA - Dataset Climático da Região Sudeste

Este notebook tem como objetivo realizar uma análise exploratória inicial do dataset climático da região Sudeste do Brasil.

Nesta etapa, ainda não vamos remover colunas do dataset.

O objetivo é entender:

- estrutura geral dos dados;
- quantidade de linhas e colunas;
- período disponível;
- estados e estações presentes;
- variáveis climáticas disponíveis;
- quantidade de valores ausentes;
- presença do valor `-9999`, usado no dataset para indicar dado faltante;
- estatísticas básicas das variáveis numéricas;
- comportamento inicial da temperatura, umidade, precipitação, vento, pressão e radiação.

A partir desta EDA, será possível decidir quais colunas, estados e estações fazem mais sentido para a etapa de modelagem.

In [55]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col, count, countDistinct, when, isnan,
    min, max, avg, stddev, round,
    to_date, concat_ws, to_timestamp,
    year, month, dayofmonth, hour
)

from pyspark.sql.functions import sum as spark_sum

from pyspark.sql.types import (
    DoubleType, FloatType, IntegerType, LongType, ShortType
)

import matplotlib.pyplot as plt

## 1. Criando a sessão Spark

In [38]:
spark = (
    SparkSession.builder
    .appName("EDA_Weather_Surface_Brazil_Southeast")
    .getOrCreate()
)

spark

## 2. Carregando o dataset em Parquet

A análise será feita a partir do arquivo Parquet, pois ele é mais eficiente que CSV para trabalhar com Spark.

O CSV original é muito grande, então o ideal é converter uma vez para Parquet e depois usar o Parquet nas próximas etapas.

In [39]:
parquet_path = "/home/jovyan/work/data/processed/weather_southeast_parquet"

df = spark.read.parquet(parquet_path)
df.createOrReplaceTempView("weather_raw")

## 3. Visualização inicial

Vamos visualizar as primeiras linhas para entender o formato geral dos dados.

In [40]:
spark.sql("""
    SELECT *
    FROM weather_raw
    LIMIT 5
""").show(truncate=False)

+-----+----------+-------------------+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+------+-----+---------+------------+------------+------------+------+
|index|Data      |Hora               |PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)|PRESSÃO ATMOSFERICA MIN. NA HOR

## 4. Estrutura do dataset

Aqui vamos observar os nomes das colunas e os tipos de dados identificados pelo Spark.

In [41]:
df.printSchema()

root
 |-- index: integer (nullable = true)
 |-- Data: date (nullable = true)
 |-- Hora: timestamp (nullable = true)
 |-- PRECIPITAÇÃO TOTAL, HORÁRIO (mm): double (nullable = true)
 |-- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB): double (nullable = true)
 |-- RADIACAO GLOBAL (Kj/m²): integer (nullable = true)
 |-- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C): double (nullable = true)
 |-- TEMPERATURA DO PONTO DE ORVALHO (°C): double (nullable = true)
 |-- TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C): double (nullable = true)
 |-- UMIDADE REL. MAX. NA HORA ANT. (AUT) (%): integer (nullable = t

In [42]:
print("Quantidade de colunas:", len(df.columns))

print("\nColunas do dataset:")
for c in df.columns:
    print(c)

Quantidade de colunas: 27

Colunas do dataset:
index
Data
Hora
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)
RADIACAO GLOBAL (Kj/m²)
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)
TEMPERATURA DO PONTO DE ORVALHO (°C)
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)
UMIDADE RELATIVA DO AR, HORARIA (%)
VENTO, DIREÇÃO HORARIA (gr) (° (gr))
VENTO, RAJADA MAXIMA (m/s)
VENTO, VELOCIDADE HORARIA (m/s)
region
state
station
station_code
latitude
longitude
height


## 5. Quantidade de registros

O `count()` percorre o dataset inteiro, então pode demorar um pouco.

Como esta informação é importante para a EDA, vamos executar uma vez e guardar o resultado.

In [43]:
resultado_dimensao = spark.sql("""
    SELECT COUNT(*) AS total_linhas
    FROM weather_raw
""").collect()[0]

total_linhas = resultado_dimensao["total_linhas"]
total_colunas = len(df.columns)

print(f"Total de linhas: {total_linhas}")
print(f"Total de colunas: {total_colunas}")

Total de linhas: 15345216
Total de colunas: 27


## 6. Separando colunas numéricas e categóricas

Essa separação ajuda a entender quais colunas podem ser usadas para estatísticas e quais representam identificação, localização ou categorias.

In [44]:
tipos_numericos = (DoubleType, FloatType, IntegerType, LongType, ShortType)

colunas_numericas = [
    campo.name
    for campo in df.schema.fields
    if isinstance(campo.dataType, tipos_numericos)
]

colunas_nao_numericas = [
    campo.name
    for campo in df.schema.fields
    if campo.name not in colunas_numericas
]

print("Colunas numéricas:")
for c in colunas_numericas:
    print("-", c)

print("\nColunas não numéricas:")
for c in colunas_nao_numericas:
    print("-", c)

Colunas numéricas:
- index
- PRECIPITAÇÃO TOTAL, HORÁRIO (mm)
- PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)
- PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)
- PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)
- RADIACAO GLOBAL (Kj/m²)
- TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)
- TEMPERATURA DO PONTO DE ORVALHO (°C)
- TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)
- TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)
- TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)
- TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)
- UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)
- UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)
- UMIDADE RELATIVA DO AR, HORARIA (%)
- VENTO, DIREÇÃO HORARIA (gr) (° (gr))
- VENTO, RAJADA MAXIMA (m/s)
- VENTO, VELOCIDADE HORARIA (m/s)
- latitude
- longitude
- height

Colunas não numéricas:
- Data
- Hora
- region
- state
- station
- station_code


## 7. Tratamento inicial de data e hora

No dataset, a data e a hora estão em colunas separadas:

- `Data`
- `Hora`

Vamos criar uma coluna `data_hora` para facilitar análises temporais.

Essa etapa não remove nenhuma coluna original.

In [45]:
df_eda = (
    df
    .withColumn("data", to_date(col("Data"), "yyyy-MM-dd"))
    .withColumn("data_hora", to_timestamp(concat_ws(" ", col("Data"), col("Hora")), "yyyy-MM-dd HH:mm"))
    .withColumn("ano", year(col("data_hora")))
    .withColumn("mes", month(col("data_hora")))
    .withColumn("dia", dayofmonth(col("data_hora")))
    .withColumn("hora_num", hour(col("data_hora")))
)

## 8. Verificando o período dos dados

Vamos identificar a menor e a maior data disponíveis no dataset.

In [46]:
spark.sql("""
    SELECT
        MIN(data) AS data_inicial,
        MAX(data) AS data_final
    FROM weather_sudeste
""").show(truncate=False)

+------------+----------+
|data_inicial|data_final|
+------------+----------+
|2000-05-07  |2021-04-30|
+------------+----------+



## 9. Estados disponíveis

Como o dataset é da região Sudeste, vamos verificar quais estados aparecem e quantos registros existem para cada um.

In [47]:
spark.sql("""
    SELECT
        region,
        state,
        COUNT(*) AS total_registros
    FROM weather_sudeste
    GROUP BY region, state
    ORDER BY state
""").show(truncate=False)

+------+-----+---------------+
|region|state|total_registros|
+------+-----+---------------+
|SE    |ES   |1211856        |
|SE    |MG   |7350864        |
|SE    |RJ   |2493936        |
|SE    |SP   |4288560        |
+------+-----+---------------+



## 10. Estações meteorológicas disponíveis

Vamos verificar quantas estações existem por estado.

Essa etapa é importante porque, futuramente, podemos escolher estações representando litoral, serra, capital, interior ou áreas urbanas.

In [48]:
spark.sql("""
    SELECT
        state,
        COUNT(DISTINCT station) AS total_estacoes
    FROM weather_sudeste
    GROUP BY state
    ORDER BY state
""").show(truncate=False)

+-----+--------------+
|state|total_estacoes|
+-----+--------------+
|ES   |13            |
|MG   |68            |
|RJ   |36            |
|SP   |43            |
+-----+--------------+



## 11. Lista de estações por estado

Aqui vamos listar as estações meteorológicas e suas coordenadas.

Essa etapa ajuda a entender quais cidades/estações existem no dataset.

In [49]:
spark.sql("""
    SELECT DISTINCT
        state,
        station,
        station_code,
        latitude,
        longitude,
        height
    FROM weather_sudeste
    ORDER BY state, station
""").show(300, truncate=False)

+-----+------------------------------------+------------+------------+------------+-------+
|state|station                             |station_code|latitude    |longitude   |height |
+-----+------------------------------------+------------+------------+------------+-------+
|ES   |AFONSO CLAUDIO                      |A657        |-20.10416666|-41.10694444|520.0  |
|ES   |AFONSO CLAUDIO                      |A657        |-20.10416666|-41.10694444|507.48 |
|ES   |ALEGRE                              |A617        |-20.75055555|-41.48888888|138.0  |
|ES   |ALFREDO CHAVES                      |A615        |-20.63638888|-40.74138888|35.0   |
|ES   |ALFREDO CHAVES                      |A615        |-20.636526  |-40.741818  |14.19  |
|ES   |ALFREDO CHAVES                      |A615        |-20.63638888|-40.74194444|14.19  |
|ES   |ECOPORANGA                          |A631        |-18.29166666|-40.73638888|224.0  |
|ES   |ECOPORANGA                          |A631        |-18.29138888|-40.736388

## 12. Foco inicial em São Paulo

Ainda não vamos filtrar o dataset definitivamente.

Mas como existe interesse em analisar futuramente o estado de São Paulo, vamos apenas visualizar quais estações de SP existem.

In [50]:
spark.sql("""
    SELECT DISTINCT
        station,
        station_code,
        latitude,
        longitude,
        height
    FROM weather_sudeste
    WHERE state = 'SP'
    ORDER BY station
""").show(300, truncate=False)

+----------------------+------------+------------+------------+-------+
|station               |station_code|latitude    |longitude   |height |
+----------------------+------------+------------+------------+-------+
|ARIRANHA              |A736        |-21.13305554|-48.84027777|525.0  |
|ARIRANHA              |A736        |-21.13305554|-48.84055555|525.44 |
|AVARE                 |A725        |-23.09972221|-48.94555555|775.0  |
|AVARE                 |A725        |-23.10166666|-48.9411111 |776.36 |
|BARRA BONITA          |A741        |-22.37083332|-48.55722222|544.0  |
|BARRA BONITA          |A741        |-22.4711111 |-48.5575    |533.68 |
|BARRA DO TURVO        |A746        |-24.96305555|-48.41638888|667.0  |
|BARRA DO TURVO        |A746        |-24.96277777|-48.41638888|659.89 |
|BARRETOS              |A748        |-20.55888888|-48.54472221|533.0  |
|BARRETOS              |A748        |-20.55916666|-48.54499999|534.36 |
|BARUERI               |A755        |-23.52333332|-46.86916666|7

## 13. Valores ausentes representados por `-9999`

Este dataset usa o valor `-9999` para representar dados ausentes ou inválidos.

Isso é muito importante, porque o Spark não entende `-9999` como nulo automaticamente.

Se calcularmos médias sem tratar esse valor, os resultados ficarão errados.

In [56]:
expressoes_9999 = []

for c in colunas_numericas:
    coluna_segura = f"`{c}`"

    expressoes_9999.append(
        spark_sum(
            when(col(coluna_segura) == -9999, 1).otherwise(0)
        ).alias(c)
    )

df_9999 = df_eda.select(*expressoes_9999)

df_9999.show(truncate=False)

+-----+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+--------+---------+------+
|index|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)|PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)|RADIACAO GLOBAL (Kj/m²)|TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)|TEMPERATURA DO PON

## 14. Percentual de `-9999` por coluna numérica

Agora vamos calcular o percentual de valores `-9999` em cada coluna numérica.

In [58]:
expressoes_percentual_9999 = []

for c in colunas_numericas:
    expressoes_percentual_9999.append(
        round(
            (
                spark_sum(
                    when(col(f"`{c}`") == -9999, 1).otherwise(0)
                ) / total_linhas
            ) * 100,
            2
        ).alias(c)
    )

df_percentual_9999 = df_eda.select(*expressoes_percentual_9999)

df_percentual_9999.show(truncate=False)

+-----+--------------------------------+-----------------------------------------------------+-----------------------------------------------+------------------------------------------------+-----------------------+--------------------------------------------+------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------------+------------------------------------------------+----------------------------------------+----------------------------------------+-----------------------------------+------------------------------------+--------------------------+-------------------------------+--------+---------+------+
|index|PRECIPITAÇÃO TOTAL, HORÁRIO (mm)|PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)|PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)|PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)|RADIACAO GLOBAL (Kj/m²)|TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)|TEMPERATURA DO PON

## 15. Criando uma versão tratada para análise

Vamos criar uma nova versão do dataframe chamada `df_tratado`.

Nela, os valores `-9999` serão convertidos para `null`.

Isso não remove nenhuma coluna do dataset original. Apenas permite que as estatísticas sejam calculadas corretamente.

In [60]:
df_tratado = df_eda

for c in colunas_numericas:
    df_tratado = df_tratado.withColumn(
        c,
        when(col(f"`{c}`") == -9999, None).otherwise(col(f"`{c}`"))
    )

In [61]:
df_tratado.createOrReplaceTempView("weather_sudeste_tratado")

## 16. Cache da base tratada

Como a base tratada será usada em várias análises, vamos usar `cache()`.

Isso ajuda a evitar que o Spark recalcule tudo várias vezes.

In [64]:
df_tratado.createOrReplaceTempView("weather_sudeste_tratado")

ConnectionRefusedError: [Errno 111] Connection refused

## 17. Valores nulos após tratamento

Agora que os `-9999` foram convertidos para `null`, vamos verificar a quantidade de nulos por coluna.

In [ ]:
expressoes_nulos = []

for c in df_tratado.columns:
    expressoes_nulos.append(
        count(when(col(c).isNull(), c)).alias(c)
    )

df_nulos = df_tratado.select(expressoes_nulos)

df_nulos.show(truncate=False)

## 18. Percentual de nulos após tratamento

In [ ]:
expressoes_percentual_nulos = []

for c in df_tratado.columns:
    expressoes_percentual_nulos.append(
        round(
            (count(when(col(c).isNull(), c)) / total_linhas) * 100,
            2
        ).alias(c)
    )

df_percentual_nulos = df_tratado.select(expressoes_percentual_nulos)

df_percentual_nulos.show(truncate=False)

## 19. Registros por ano

Essa análise mostra se todos os anos possuem uma quantidade semelhante de registros.

In [ ]:
spark.sql("""
    SELECT
        ano,
        COUNT(*) AS total_registros
    FROM weather_sudeste_tratado
    GROUP BY ano
    ORDER BY ano
""").show(100, truncate=False)

## 20. Registros por mês

Essa análise ajuda a verificar a distribuição dos dados ao longo dos meses.

In [ ]:
spark.sql("""
    SELECT
        ano,
        COUNT(*) AS total_registros
    FROM weather_sudeste_tratado
    GROUP BY ano
    ORDER BY ano
""").show(100, truncate=False)

## 21. Estatísticas descritivas das colunas numéricas

Agora que `-9999` virou `null`, as estatísticas fazem mais sentido.

Vamos calcular média, mínimo, máximo e desvio padrão.

In [ ]:
estatisticas = []

for c in colunas_numericas:
    estatisticas.extend([
        round(avg(c), 2).alias(f"media_{c}"),
        round(min(c), 2).alias(f"min_{c}"),
        round(max(c), 2).alias(f"max_{c}"),
        round(stddev(c), 2).alias(f"desvio_{c}")
    ])

df_tratado.select(estatisticas).show(truncate=False)

## 22. Identificando colunas relacionadas a temperatura

Vamos localizar automaticamente as colunas de temperatura disponíveis no dataset.

In [ ]:
colunas_temperatura = [
    c for c in df_tratado.columns
    if "TEMPERATURA" in c.upper()
]

print("Colunas relacionadas à temperatura:")
for c in colunas_temperatura:
    print("-", c)

## 23. Escolhendo a coluna principal de temperatura

Para uma primeira análise, vamos usar a temperatura horária do ar.

No dataset, essa coluna é:

`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`

In [ ]:
spark.sql("""
    SELECT
        `TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`
    FROM weather_sudeste_tratado
    LIMIT 10
""").show()

## 24. Temperatura média por estado

Vamos comparar a temperatura média entre os estados do Sudeste.

In [ ]:
temp_estado = spark.sql("""
    SELECT
        state,
        ROUND(AVG(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_media,
        ROUND(MIN(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_minima,
        ROUND(MAX(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_maxima,
        COUNT(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`) AS registros_validos
    FROM weather_sudeste_tratado
    GROUP BY state
    ORDER BY state
""")

temp_estado.show(truncate=False)

In [ ]:
dados_estado = temp_estado.collect()

estados = [linha["state"] for linha in dados_estado]
temperaturas = [linha["temperatura_media"] for linha in dados_estado]

plt.figure(figsize=(8, 5))
plt.bar(estados, temperaturas)
plt.title("Temperatura média por estado")
plt.xlabel("Estado")
plt.ylabel("Temperatura média (°C)")
plt.show()

## 25. Temperatura média por ano

Vamos observar a evolução da temperatura média ao longo dos anos.

In [ ]:
temp_ano = spark.sql("""
    SELECT
        ano,
        ROUND(AVG(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_media,
        COUNT(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`) AS registros_validos
    FROM weather_sudeste_tratado
    GROUP BY ano
    ORDER BY ano
""")

temp_ano.show(100, truncate=False)

In [ ]:
dados_ano = temp_ano.collect()

anos = [linha["ano"] for linha in dados_ano]
temperaturas = [linha["temperatura_media"] for linha in dados_ano]

plt.figure(figsize=(10, 5))
plt.plot(anos, temperaturas, marker="o")
plt.title("Temperatura média anual - Sudeste")
plt.xlabel("Ano")
plt.ylabel("Temperatura média (°C)")
plt.grid(True)
plt.show()

## 26. Temperatura média por estado e ano

Essa análise ajuda a comparar a evolução da temperatura entre os estados.

In [ ]:
temp_estado_ano = spark.sql("""
    SELECT
        ano,
        state,
        ROUND(AVG(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_media
    FROM weather_sudeste_tratado
    GROUP BY ano, state
    ORDER BY ano, state
""")

temp_estado_ano.show(200, truncate=False)

In [ ]:
dados_estado_ano = temp_estado_ano.collect()

estados = sorted(list(set([linha["state"] for linha in dados_estado_ano])))

plt.figure(figsize=(12, 6))

for estado in estados:
    dados_filtrados = [
        linha for linha in dados_estado_ano
        if linha["state"] == estado
    ]

    anos = [linha["ano"] for linha in dados_filtrados]
    temperaturas = [linha["temperatura_media"] for linha in dados_filtrados]

    plt.plot(anos, temperaturas, marker="o", label=estado)

plt.title("Temperatura média anual por estado")
plt.xlabel("Ano")
plt.ylabel("Temperatura média (°C)")
plt.legend()
plt.grid(True)
plt.show()

## 27. Temperatura média por estação de São Paulo

Como existe interesse em analisar futuramente o estado de São Paulo, vamos observar a temperatura média por estação meteorológica em SP.

Ainda não estamos filtrando definitivamente a base; é apenas uma análise exploratória.

In [ ]:
temp_estacoes_sp = spark.sql("""
    SELECT
        station,
        station_code,
        latitude,
        longitude,
        height,
        ROUND(AVG(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_media,
        ROUND(MIN(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_minima,
        ROUND(MAX(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_maxima,
        COUNT(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`) AS registros_validos
    FROM weather_sudeste_tratado
    WHERE state = 'SP'
    GROUP BY station, station_code, latitude, longitude, height
    ORDER BY temperatura_media
""")

temp_estacoes_sp.show(300, truncate=False)

## 28. Temperatura média por altitude em SP

A coluna `height` representa a altitude da estação.

Essa análise é importante porque a temperatura pode variar bastante entre litoral, planalto, áreas urbanas e regiões de serra.

In [ ]:
temp_altitude_sp = spark.sql("""
    SELECT
        station,
        station_code,
        height,
        ROUND(AVG(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`), 2) AS temperatura_media,
        COUNT(`TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)`) AS registros_validos
    FROM weather_sudeste_tratado
    WHERE state = 'SP'
    GROUP BY station, station_code, height
    ORDER BY height
""")

temp_altitude_sp.show(300, truncate=False)

## 29. Distribuição da temperatura usando PySpark

Para analisar a distribuição da temperatura, não vamos converter o dataset para Pandas.

Vamos usar recursos do próprio PySpark, como histograma em RDD e quantis aproximados.

## 30. Histograma da temperatura

Vamos observar a distribuição da temperatura horária na amostra.

In [ ]:
temperaturas_validas = spark.sql("""
    SELECT
        `TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)` AS temperatura
    FROM weather_sudeste_tratado
    WHERE `TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)` IS NOT NULL
""")

In [ ]:
dados_temp = (
    temperaturas_validas
    .select("temperatura")
    .rdd
    .flatMap(lambda linha: linha)
)

histograma = dados_temp.histogram(50)

bins = histograma[0]
frequencias = histograma[1]

largura_barra = bins[1] - bins[0]

plt.figure(figsize=(10, 5))
plt.bar(
    bins[:-1],
    frequencias,
    width=largura_barra
)
plt.title("Distribuição da temperatura do ar")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frequência")
plt.show()

## 31. Boxplot da temperatura

O boxplot ajuda a visualizar dispersão, mediana e possíveis valores extremos.

In [ ]:
quartis_temp = df_tratado.approxQuantile(
    col_temp,
    [0.0, 0.25, 0.5, 0.75, 1.0],
    0.01
)

print("Resumo aproximado da temperatura do ar:")
print(f"Mínimo: {quartis_temp[0]}")
print(f"1º quartil: {quartis_temp[1]}")
print(f"Mediana: {quartis_temp[2]}")
print(f"3º quartil: {quartis_temp[3]}")
print(f"Máximo: {quartis_temp[4]}")

## 32. Colunas climáticas principais

Vamos identificar colunas relacionadas a grupos climáticos importantes:

- temperatura;
- umidade;
- precipitação;
- pressão;
- vento;
- radiação.

In [ ]:
grupos_climaticos = {
    "temperatura": ["TEMPERATURA"],
    "umidade": ["UMIDADE"],
    "precipitacao": ["PRECIPITAÇÃO"],
    "pressao": ["PRESSAO", "PRESSÃO"],
    "vento": ["VENTO"],
    "radiacao": ["RADIACAO", "RADIAÇÃO"]
}

for grupo, palavras in grupos_climaticos.items():
    print(f"\n{grupo.upper()}:")
    for c in df_tratado.columns:
        if any(p in c.upper() for p in palavras):
            print("-", c)

## 33. Conclusões iniciais da EDA

Com base nesta análise, os próximos passos serão:

1. verificar quais colunas possuem muitos dados ausentes;
2. escolher variáveis climáticas úteis para o projeto;
3. decidir se o foco será em todos os estados do Sudeste ou apenas em São Paulo;
4. caso o foco seja São Paulo, escolher estações representando diferentes contextos geográficos:
   - litoral;
   - serra/altitude;
   - capital/região urbana;
   - interior;
5. definir uma coluna alvo para classificação ou regressão;
6. preparar uma base limpa para modelagem em PySpark.

In [ ]:
df_tratado.unpersist()